# Similarity Correlation

Check whether embedding-space cosine similarity correlates with structural similarity
computed independently from reaction fingerprints (Pearson/Spearman), for a random
sample of reaction pairs.

> Requires `uv sync --extra all`, plus:
> - `data/embeddings/medium/{embeddings.npy,smarts.txt}` — download from [Zenodo](https://doi.org/10.5281/zenodo.22645328), or regenerate (see README's Data pipeline and Training sections).

In [ ]:
from pathlib import Path
import itertools

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, linregress
from rdkit.Chem import rdChemReactions
from rdkit.DataStructs import TanimotoSimilarity

## Load embeddings

In [ ]:
EMB_PATH    = "data/embeddings/medium/embeddings.npy"
SMARTS_PATH = "data/embeddings/medium/smarts.txt"

embeddings  = np.load(EMB_PATH).astype(np.float32)
smarts_list = Path(SMARTS_PATH).read_text().splitlines()

print(f"Loaded {embeddings.shape[0]:,} embeddings  dim={embeddings.shape[1]}")

## Sample reactions and compute structural fingerprints

Structural similarity is computed independently of the model, from RDKit's structural
reaction fingerprint (4096-bit, built from the reactant/product substructures) —
unrelated to the SMARTS tokens the model actually sees.

In [ ]:
N_REACTIONS = 1000   # → ~500k pairs; increase for a smoother estimate (slower)
SEED = 42

rng = np.random.default_rng(SEED)
idx = rng.choice(len(smarts_list), size=N_REACTIONS, replace=False)
idx.sort()

def reaction_fingerprint(smarts: str):
    rxn = rdChemReactions.ReactionFromSmarts(smarts)
    if rxn is None:
        return None
    return rdChemReactions.CreateStructuralFingerprintForReaction(rxn)

fps = [reaction_fingerprint(smarts_list[i]) for i in idx]
valid = [fp is not None for fp in fps]
print(f"Sampled {N_REACTIONS} reactions — {sum(valid)}/{N_REACTIONS} fingerprinted successfully")

## Pairwise similarities

In [ ]:
# Structural Tanimoto similarity (RDKit fingerprints)
pairs = list(itertools.combinations(range(N_REACTIONS), 2))
tanimoto = np.array([
    TanimotoSimilarity(fps[a], fps[b]) if valid[a] and valid[b] else np.nan
    for a, b in pairs
], dtype=np.float32)

# Embedding cosine similarity (vectorised)
emb = embeddings[idx]
normed = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10)
sim_matrix = normed @ normed.T
i, j = np.triu_indices(N_REACTIONS, k=1)
cosine = sim_matrix[i, j]

# Drop pairs with a failed fingerprint
keep = ~np.isnan(tanimoto)
tanimoto, cosine = tanimoto[keep], cosine[keep]
print(f"{keep.sum():,} / {len(keep):,} pairs retained")

## Correlation

In [ ]:
pearson_r, pearson_p = pearsonr(tanimoto, cosine)
spearman_r, spearman_p = spearmanr(tanimoto, cosine)

print(f"Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")
print(f"Spearman ρ = {spearman_r:.4f}  (p = {spearman_p:.2e})")
print(f"N pairs    = {len(cosine):,}")

## Visualise

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

hb = ax.hexbin(tanimoto, cosine, gridsize=50, cmap="YlOrRd", mincnt=1, linewidths=0.1)
fig.colorbar(hb, ax=ax, pad=0.02, label="Pair count")

slope, intercept, *_ = linregress(tanimoto, cosine)
x_line = np.array([float(tanimoto.min()), float(tanimoto.max())])
ax.plot(x_line, slope * x_line + intercept, color="#1d4ed8", lw=1.5, ls="--", label="Linear fit")

ax.set_xlabel("Tanimoto similarity (structural reaction fingerprint)")
ax.set_ylabel("Cosine similarity (embedding space)")
ax.set_title("Embedding Similarity vs Structural Similarity", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

info = f"Pearson $r$ = {pearson_r:.3f}\nSpearman $\\rho$ = {spearman_r:.3f}\n$n$ = {len(cosine):,} pairs"
ax.text(0.97, 0.05, info, transform=ax.transAxes, ha="right", va="bottom", fontsize=9.5,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9))

ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()